In [1]:
!pip install transformers sentence-transformers faiss-cpu langchain


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 54.8 MB/s eta 0:00:00


In [3]:
knowledge = """Company Policy Manual:
- WFH Policy: All employees are eligible for a hybrid WFH schedule. Employees must be in the office on Tuesdays, Wednesdays, and Thursdays. Mondays and Fridays are optional remote days.
- PTO Policy: Full-time employees receive 20 days of Paid Time Off (PTO) per year. PTO accrues monthly.
- Tech Stack: The official backend language is Python, and the official frontend framework is React. For mobile development, we use React Native.
"""

with open("my_knowledge.txt", "w") as f:
    f.write(knowledge)

print("✅ my_knowledge.txt created!")


✅ my_knowledge.txt created!


In [10]:
with open("my_knowledge.txt", "r") as f:
    print(f.read())

Company Policy Manual:
- WFH Policy: All employees are eligible for a hybrid WFH schedule. Employees must be in the office on Tuesdays, Wednesdays, and Thursdays. Mondays and Fridays are optional remote days.
- PTO Policy: Full-time employees receive 20 days of Paid Time Off (PTO) per year. PTO accrues monthly.
- Tech Stack: The official backend language is Python, and the official frontend framework is React. For mobile development, we use React Native.



In [12]:

!pip install -q langchain langchain-text-splitters sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 17.9 MB/s eta 0:00:00


In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

# ---- STEP 2 ----
with open("my_knowledge.txt") as f:
    knowledge_text = f.read()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=20,
    length_function=len
)
chunks = text_splitter.split_text(knowledge_text)
print(f"✅ Step 2 done — {len(chunks)} chunks created")

# ---- STEP 3 ----
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(chunks)
print(f"✅ Step 3 done — {len(embeddings)} embeddings created")
print(f"Each embedding has {len(embeddings[0])} dimensions")

✅ Step 2 done — 5 chunks created


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Step 3 done — 5 embeddings created
Each embedding has 384 dimensions


In [15]:

import faiss
import numpy as np

# ---- STEP 4 ----
# Get the dimension of our vectors (384)
d = embeddings.shape[1]

# Create a FAISS index
index = faiss.IndexFlatL2(d)

# Add our embeddings to the index (must be float32)
index.add(np.array(embeddings).astype('float32'))

print(f"✅ Step 4 done — FAISS index created with {index.ntotal} vectors")

✅ Step 4 done — FAISS index created with 5 vectors


In [17]:
from transformers import pipeline

# Fixed: using 'text-generation' instead of 'text2text-generation'
generator = pipeline('text-generation', model='google/flan-t5-small')
print("✅ Generator model loaded!")

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

✅ Generator model loaded!


In [18]:
def answer_question(query):
    # 1. RETRIEVE
    query_embedding = model.encode([query]).astype('float32')

    k = 2
    distances, indices = index.search(query_embedding, k)

    retrieved_chunks = [chunks[i] for i in indices[0]]
    context = "\n\n".join(retrieved_chunks)

    # 2. AUGMENT
    prompt_template = f"""Answer the following question using only the provided context.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question:
{query}

Answer:"""

    # 3. GENERATE
    answer = generator(prompt_template, max_new_tokens=100)
    print(f"--- RETRIEVED CONTEXT ---\n{context}\n")
    print(f"--- ANSWER ---")
    return answer[0]['generated_text']


In [19]:
print(answer_question("What days can I work from home?"))
print("\n" + "="*50 + "\n")
print(answer_question("How many PTO days do I get?"))
print("\n" + "="*50 + "\n")
print(answer_question("What is the frontend framework?"))

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- RETRIEVED CONTEXT ---
Thursdays. Mondays and Fridays are optional remote days.

- WFH Policy: All employees are eligible for a hybrid WFH schedule. Employees must be in the office on Tuesdays, Wednesdays, and Thursdays. Mondays

--- ANSWER ---
Answer the following question using only the provided context.
If the answer is not in the context, say "I don't have that information."

Context:
Thursdays. Mondays and Fridays are optional remote days.

- WFH Policy: All employees are eligible for a hybrid WFH schedule. Employees must be in the office on Tuesdays, Wednesdays, and Thursdays. Mondays

Question:
What days can I work from home?

Answer:y per Sunday) instead only do those schedule sessions and


--- RETRIEVED CONTEXT ---
- PTO Policy: Full-time employees receive 20 days of Paid Time Off (PTO) per year. PTO accrues monthly.

Thursdays. Mondays and Fridays are optional remote days.

--- ANSWER ---
Answer the following question using only the provided context.
If the answer is not 

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- RETRIEVED CONTEXT ---
- Tech Stack: The official backend language is Python, and the official frontend framework is React. For mobile development, we use React Native.

Company Policy Manual:

--- ANSWER ---
Answer the following question using only the provided context.
If the answer is not in the context, say "I don't have that information."

Context:
- Tech Stack: The official backend language is Python, and the official frontend framework is React. For mobile development, we use React Native.

Company Policy Manual:

Question:
What is the frontend framework?

Answer:
